
# Neural Networks Lab – Complete Google Colab Notebook

This notebook implements the experiments from **A1 to A12** in the assignment.

The perceptron from the supplied diagram is:

\[
y = w_0(1) + w_1A + w_2B
\]

\[
Z = f(y)
\]

For the AND/XOR truth tables, the input order used throughout is:

`[A, B] = [0,0], [0,1], [1,0], [1,1]`

The code for A1–A10 is written from scratch. `sklearn` is used only where the assignment explicitly asks for `MLPClassifier` or pseudo-inverse comparison.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

print("Libraries loaded.")


## A1 – Functions: summation, activation functions and comparator

In [ ]:

def summation_unit(inputs, weights, bias=0.0):
    inputs = np.asarray(inputs, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return bias + np.dot(inputs, weights)


def step_activation(z):
    return 1 if z > 0 else 0


def bipolar_step_activation(z):
    if z > 0:
        return 1
    elif z == 0:
        return 0
    else:
        return -1


def sigmoid_activation(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -60, 60)
    return 1.0 / (1.0 + np.exp(-z))


def tanh_activation(z):
    return np.tanh(z)


def relu_activation(z):
    return np.maximum(0, z)


def leaky_relu_activation(z, alpha=0.01):
    z = np.asarray(z, dtype=float)
    return np.where(z > 0, z, alpha * z)


def comparator(target, predicted):
    return np.asarray(target) - np.asarray(predicted)


def sum_squared_error(target, predicted):
    return float(np.sum((np.asarray(target) - np.asarray(predicted)) ** 2))


print("A1 functions are defined.")
print("Sigmoid(0) =", sigmoid_activation(0))
print("Tanh(0) =", tanh_activation(0))
print("ReLU(-2), ReLU(2) =", relu_activation(-2), relu_activation(2))
print("Leaky ReLU(-2) =", leaky_relu_activation(-2))


## Activation-function plots

In [ ]:

z_values = np.linspace(-5, 5, 500)

plt.figure(figsize=(9, 6))
plt.plot(z_values, [step_activation(z) for z in z_values])
plt.title("Step Activation")
plt.xlabel("z")
plt.ylabel("f(z)")
plt.grid(True)
plt.show()

plt.figure(figsize=(9, 6))
plt.plot(z_values, [bipolar_step_activation(z) for z in z_values])
plt.title("Bipolar Step Activation")
plt.xlabel("z")
plt.ylabel("f(z)")
plt.grid(True)
plt.show()

plt.figure(figsize=(9, 6))
plt.plot(z_values, sigmoid_activation(z_values))
plt.title("Sigmoid Activation")
plt.xlabel("z")
plt.ylabel("f(z)")
plt.grid(True)
plt.show()

plt.figure(figsize=(9, 6))
plt.plot(z_values, tanh_activation(z_values))
plt.title("Tanh Activation")
plt.xlabel("z")
plt.ylabel("f(z)")
plt.grid(True)
plt.show()

plt.figure(figsize=(9, 6))
plt.plot(z_values, relu_activation(z_values))
plt.title("ReLU Activation")
plt.xlabel("z")
plt.ylabel("f(z)")
plt.grid(True)
plt.show()

plt.figure(figsize=(9, 6))
plt.plot(z_values, leaky_relu_activation(z_values))
plt.title("Leaky ReLU Activation")
plt.xlabel("z")
plt.ylabel("f(z)")
plt.grid(True)
plt.show()


## Common truth tables and perceptron training function

In [ ]:

X_gate = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

Y_AND = np.array([0, 0, 0, 1], dtype=float)
Y_XOR = np.array([0, 1, 1, 0], dtype=float)


def predict_single_perceptron(X, weights, activation):
    X = np.asarray(X, dtype=float)
    outputs = []

    for row in X:
        z = summation_unit(row, weights[1:], weights[0])
        outputs.append(activation(z))

    return np.asarray(outputs, dtype=float)


def train_perceptron(X, y, activation, initial_weights, learning_rate,
                     max_epochs=1000, convergence_error=0.002,
                     target_transform=None):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    if target_transform is not None:
        train_y = target_transform(y)
    else:
        train_y = y.copy()

    weights = np.asarray(initial_weights, dtype=float).copy()
    epoch_errors = []

    for epoch in range(1, max_epochs + 1):

        for xi, target in zip(X, train_y):
            z = summation_unit(xi, weights[1:], weights[0])
            output = activation(z)

            error = target - output

            update_vector = np.concatenate(([1.0], xi))
            weights += learning_rate * error * update_vector

        predictions = predict_single_perceptron(X, weights, activation)
        sse = sum_squared_error(train_y, predictions)
        epoch_errors.append(sse)

        if sse <= convergence_error:
            return {
                "converged": True,
                "epoch": epoch,
                "weights": weights,
                "errors": np.asarray(epoch_errors)
            }

    return {
        "converged": False,
        "epoch": max_epochs,
        "weights": weights,
        "errors": np.asarray(epoch_errors)
    }


def show_gate_results(X, target, weights, activation, display_targets=None):
    predictions = predict_single_perceptron(X, weights, activation)
    result = pd.DataFrame({
        "A": X[:, 0].astype(int),
        "B": X[:, 1].astype(int),
        "Target": target if display_targets is None else display_targets,
        "Output": predictions
    })
    print(result.to_string(index=False))
    return predictions


print("Truth tables:")
print("AND:", Y_AND.astype(int))
print("XOR:", Y_XOR.astype(int))


## A2 – AND gate using the step activation function

In [ ]:

initial_weights = np.array([10.0, 0.2, -0.75])
learning_rate = 0.05

and_step_result = train_perceptron(
    X_gate,
    Y_AND,
    step_activation,
    initial_weights,
    learning_rate,
    max_epochs=1000,
    convergence_error=0.002
)

print("Converged:", and_step_result["converged"])
print("Epochs:", and_step_result["epoch"])
print("Final weights [w0, w1, w2]:", and_step_result["weights"])

and_predictions = predict_single_perceptron(
    X_gate,
    and_step_result["weights"],
    step_activation
)

print("\nAND predictions:")
show_gate_results(X_gate, Y_AND.astype(int),
                  and_step_result["weights"], step_activation)

plt.figure(figsize=(9, 5))
plt.plot(
    range(1, len(and_step_result["errors"]) + 1),
    and_step_result["errors"],
    linewidth=2
)
plt.axhline(0.002, linestyle="--", label="Convergence threshold = 0.002")
plt.xlabel("Epoch")
plt.ylabel("Sum Squared Error")
plt.title("A2 – AND Gate: Epoch vs SSE")
plt.grid(True)
plt.legend()
plt.show()


## A3 – Compare Bipolar Step, Sigmoid and ReLU

In [ ]:

activation_experiments = {
    "Bipolar Step": (bipolar_step_activation, lambda y: np.where(y == 0, -1, 1)),
    "Sigmoid": (sigmoid_activation, None),
    "ReLU": (relu_activation, None)
}

a3_results = {}

for name, (activation, transform) in activation_experiments.items():
    result = train_perceptron(
        X_gate,
        Y_AND,
        activation,
        initial_weights,
        learning_rate,
        max_epochs=1000,
        convergence_error=0.002,
        target_transform=transform
    )

    a3_results[name] = result

    print(name)
    print("Converged:", result["converged"])
    print("Iterations:", result["epoch"])
    print("Final weights:", result["weights"])
    print("Final SSE:", result["errors"][-1])
    print()

comparison_a3 = pd.DataFrame({
    "Activation": list(a3_results.keys()),
    "Iterations": [r["epoch"] for r in a3_results.values()],
    "Converged": [r["converged"] for r in a3_results.values()],
    "Final SSE": [r["errors"][-1] for r in a3_results.values()]
})

print(comparison_a3)

plt.figure(figsize=(10, 6))
for name, result in a3_results.items():
    plt.plot(
        range(1, len(result["errors"]) + 1),
        result["errors"],
        label=name
    )

plt.axhline(0.002, linestyle="--", label="Threshold = 0.002")
plt.xlabel("Epoch")
plt.ylabel("SSE")
plt.title("A3 – Error vs Epoch for Different Activations")
plt.grid(True)
plt.legend()
plt.show()


## A4 – Effect of learning rate on AND gate convergence

In [ ]:

learning_rates = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

a4_rows = []

for lr in learning_rates:
    result = train_perceptron(
        X_gate,
        Y_AND,
        step_activation,
        initial_weights,
        lr,
        max_epochs=1000,
        convergence_error=0.002
    )

    a4_rows.append({
        "Learning Rate": lr,
        "Iterations": result["epoch"],
        "Converged": result["converged"],
        "Final SSE": result["errors"][-1]
    })

a4_results = pd.DataFrame(a4_rows)
print(a4_results.to_string(index=False))

plt.figure(figsize=(9, 6))
plt.plot(a4_results["Learning Rate"], a4_results["Iterations"], marker="o")
plt.xlabel("Learning Rate")
plt.ylabel("Iterations to Converge")
plt.title("A4 – Learning Rate vs Iterations")
plt.grid(True)
plt.show()


## A5 – Repeat A1–A3 for XOR

In [ ]:

xor_initial_weights = np.array([10.0, 0.2, -0.75])

xor_step = train_perceptron(
    X_gate, Y_XOR, step_activation,
    xor_initial_weights, 0.05,
    max_epochs=1000, convergence_error=0.002
)

xor_sigmoid = train_perceptron(
    X_gate, Y_XOR, sigmoid_activation,
    xor_initial_weights, 0.05,
    max_epochs=1000, convergence_error=0.002
)

xor_relu = train_perceptron(
    X_gate, Y_XOR, relu_activation,
    xor_initial_weights, 0.05,
    max_epochs=1000, convergence_error=0.002
)

a5_results = {
    "Step": xor_step,
    "Sigmoid": xor_sigmoid,
    "ReLU": xor_relu
}

a5_table = pd.DataFrame({
    "Activation": list(a5_results.keys()),
    "Iterations": [r["epoch"] for r in a5_results.values()],
    "Converged": [r["converged"] for r in a5_results.values()],
    "Final SSE": [r["errors"][-1] for r in a5_results.values()]
})

print(a5_table.to_string(index=False))

for name, result in a5_results.items():
    print("\n", name)
    print("Final weights:", result["weights"])
    show_gate_results(
        X_gate,
        Y_XOR.astype(int),
        result["weights"],
        {"Step": step_activation, "Sigmoid": sigmoid_activation, "ReLU": relu_activation}[name]
    )

plt.figure(figsize=(10, 6))
for name, result in a5_results.items():
    plt.plot(
        range(1, len(result["errors"]) + 1),
        result["errors"],
        label=name
    )

plt.axhline(0.002, linestyle="--", label="Threshold = 0.002")
plt.xlabel("Epoch")
plt.ylabel("SSE")
plt.title("A5 – XOR: Error vs Epoch")
plt.grid(True)
plt.legend()
plt.show()

print("\nImportant: XOR cannot be represented by a single linear perceptron.")


## A6 – Customer transaction classification using a custom sigmoid perceptron

In [ ]:

customer_data = pd.DataFrame({
    "Customer": ["C_1","C_2","C_3","C_4","C_5","C_6","C_7","C_8","C_9","C_10"],
    "Candies": [20,16,27,19,24,22,15,18,21,16],
    "Mangoes": [6,3,6,1,4,1,4,4,1,2],
    "Milk_Packets": [2,6,2,2,2,5,2,2,4,4],
    "Payment": [386,289,393,110,280,167,271,274,148,198],
    "High_Value": ["Yes","Yes","Yes","No","Yes","No","Yes","Yes","No","No"]
})

customer_data


In [ ]:

X_customer = customer_data[["Candies", "Mangoes", "Milk_Packets", "Payment"]].values.astype(float)
y_customer = (customer_data["High_Value"] == "Yes").astype(float).values


X_min = X_customer.min(axis=0)
X_max = X_customer.max(axis=0)
X_customer_norm = (X_customer - X_min) / (X_max - X_min)

X_customer_bias = np.c_[np.ones(len(X_customer_norm)), X_customer_norm]


def train_sigmoid_perceptron(X, y, learning_rate=5.0,
                             max_epochs=10000, convergence_error=0.002):
    weights = np.zeros(X.shape[1], dtype=float)
    errors = []

    for epoch in range(1, max_epochs + 1):
        for xi, target in zip(X, y):
            z = np.dot(xi, weights)
            output = sigmoid_activation(z)
            error = target - output

            delta = error * output * (1 - output)
            weights += learning_rate * delta * xi

        outputs = sigmoid_activation(X @ weights)
        sse = sum_squared_error(y, outputs)
        errors.append(sse)

        if sse <= convergence_error:
            return weights, np.asarray(errors), epoch, True, outputs

    outputs = sigmoid_activation(X @ weights)
    return weights, np.asarray(errors), max_epochs, False, outputs


customer_weights, customer_errors, customer_epochs, customer_converged, customer_outputs = train_sigmoid_perceptron(
    X_customer_bias,
    y_customer,
    learning_rate=5.0,
    max_epochs=10000,
    convergence_error=0.002
)

customer_predictions = (customer_outputs >= 0.5).astype(int)

print("Converged:", customer_converged)
print("Epochs:", customer_epochs)
print("Weights:", customer_weights)
print("Final SSE:", customer_errors[-1])

customer_result = customer_data[["Customer", "High_Value"]].copy()
customer_result["Probability"] = customer_outputs
customer_result["Predicted"] = np.where(customer_predictions == 1, "Yes", "No")

print(customer_result.to_string(index=False))

accuracy_customer = np.mean(customer_predictions == y_customer)
print("\nTraining accuracy:", accuracy_customer)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(customer_errors) + 1), customer_errors)
plt.axhline(0.002, linestyle="--", label="Threshold = 0.002")
plt.xlabel("Epoch")
plt.ylabel("SSE")
plt.title("A6 – Customer Data: Epoch vs SSE")
plt.grid(True)
plt.legend()
plt.show()


## A7 – Compare custom perceptron learning with matrix pseudo-inverse

In [ ]:

pseudo_inverse_weights = np.linalg.pinv(X_customer_bias) @ y_customer
pseudo_inverse_output = X_customer_bias @ pseudo_inverse_weights
pseudo_inverse_prediction = (pseudo_inverse_output >= 0.5).astype(int)

a7_result = customer_data[["Customer", "High_Value"]].copy()
a7_result["Perceptron_Probability"] = customer_outputs
a7_result["Perceptron_Prediction"] = np.where(customer_predictions == 1, "Yes", "No")
a7_result["PseudoInverse_Output"] = pseudo_inverse_output
a7_result["PseudoInverse_Prediction"] = np.where(
    pseudo_inverse_prediction == 1, "Yes", "No"
)

print("Pseudo-inverse weights:")
print(pseudo_inverse_weights)

print("\nComparison:")
print(a7_result.to_string(index=False))

perceptron_acc = np.mean(customer_predictions == y_customer)
pseudo_acc = np.mean(pseudo_inverse_prediction == y_customer)

print("\nPerceptron accuracy:", perceptron_acc)
print("Pseudo-inverse accuracy:", pseudo_acc)
print("Perceptron SSE:", sum_squared_error(y_customer, customer_outputs))
print("Pseudo-inverse SSE:", sum_squared_error(y_customer, pseudo_inverse_output))


## A8/A9 – Back-propagation network from scratch

In [ ]:

def backpropagation_train(
    X,
    y,
    n_hidden=2,
    n_outputs=1,
    learning_rate=0.05,
    max_epochs=1000,
    convergence_error=0.002,
    seed=42,
    init_scale=0.5
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    if y.ndim == 1:
        y = y.reshape(-1, 1)

    rng = np.random.default_rng(seed)

    W_input_hidden = rng.uniform(
        -init_scale, init_scale,
        size=(X.shape[1], n_hidden)
    )

    b_hidden = rng.uniform(
        -init_scale, init_scale,
        size=(1, n_hidden)
    )

    W_hidden_output = rng.uniform(
        -init_scale, init_scale,
        size=(n_hidden, n_outputs)
    )

    b_output = rng.uniform(
        -init_scale, init_scale,
        size=(1, n_outputs)
    )

    errors = []

    for epoch in range(1, max_epochs + 1):

        for xi, target in zip(X, y):
            xi = xi.reshape(1, -1)
            target = target.reshape(1, -1)

            hidden = sigmoid_activation(
                xi @ W_input_hidden + b_hidden
            )

            output = sigmoid_activation(
                hidden @ W_hidden_output + b_output
            )

            output_error = target - output

            delta_output = (
                output_error * output * (1 - output)
            )

            delta_hidden = (
                hidden * (1 - hidden)
                * (delta_output @ W_hidden_output.T)
            )

            W_hidden_output += (
                learning_rate * hidden.T @ delta_output
            )

            b_output += learning_rate * delta_output

            W_input_hidden += (
                learning_rate * xi.T @ delta_hidden
            )

            b_hidden += learning_rate * delta_hidden

        hidden_all = sigmoid_activation(
            X @ W_input_hidden + b_hidden
        )

        output_all = sigmoid_activation(
            hidden_all @ W_hidden_output + b_output
        )

        sse = sum_squared_error(y, output_all)
        errors.append(sse)

        if sse <= convergence_error:
            return {
                "converged": True,
                "epoch": epoch,
                "W_input_hidden": W_input_hidden,
                "b_hidden": b_hidden,
                "W_hidden_output": W_hidden_output,
                "b_output": b_output,
                "errors": np.asarray(errors),
                "outputs": output_all
            }

    return {
        "converged": False,
        "epoch": max_epochs,
        "W_input_hidden": W_input_hidden,
        "b_hidden": b_hidden,
        "W_hidden_output": W_hidden_output,
        "b_output": b_output,
        "errors": np.asarray(errors),
        "outputs": output_all
    }


## A8 – 2-2-1 network for AND using sigmoid back-propagation

In [ ]:

and_bp = backpropagation_train(
    X_gate,
    Y_AND,
    n_hidden=2,
    n_outputs=1,
    learning_rate=0.05,
    max_epochs=1000,
    convergence_error=0.002,
    seed=42
)

print("Converged:", and_bp["converged"])
print("Epochs:", and_bp["epoch"])
print("Final SSE:", and_bp["errors"][-1])

print("\nW_input_hidden:")
print(and_bp["W_input_hidden"])

print("\nb_hidden:")
print(and_bp["b_hidden"])

print("\nW_hidden_output:")
print(and_bp["W_hidden_output"])

print("\nb_output:")
print(and_bp["b_output"])

print("\nAND output probabilities:")
print(and_bp["outputs"].ravel())

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(and_bp["errors"]) + 1), and_bp["errors"])
plt.axhline(0.002, linestyle="--", label="Threshold = 0.002")
plt.xlabel("Epoch")
plt.ylabel("SSE")
plt.title("A8 – Backpropagation on AND")
plt.grid(True)
plt.legend()
plt.show()


## A9 – 2-2-1 network for XOR using sigmoid back-propagation

In [ ]:

xor_bp = backpropagation_train(
    X_gate,
    Y_XOR,
    n_hidden=2,
    n_outputs=1,
    learning_rate=0.05,
    max_epochs=1000,
    convergence_error=0.002,
    seed=42,
    init_scale=0.5
)

print("Converged:", xor_bp["converged"])
print("Epochs:", xor_bp["epoch"])
print("Final SSE:", xor_bp["errors"][-1])

print("\nXOR output probabilities:")
print(xor_bp["outputs"].ravel())

xor_bp_predictions = (xor_bp["outputs"].ravel() >= 0.5).astype(int)

print("\nPredictions:")
print(xor_bp_predictions)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(xor_bp["errors"]) + 1), xor_bp["errors"])
plt.axhline(0.002, linestyle="--", label="Threshold = 0.002")
plt.xlabel("Epoch")
plt.ylabel("SSE")
plt.title("A9 – Backpropagation on XOR")
plt.grid(True)
plt.legend()
plt.show()


## A10 – Two output nodes for AND gate

In [ ]:

Y_AND_2OUT = np.array([
    [1, 0],
    [1, 0],
    [1, 0],
    [0, 1]
], dtype=float)

and_2out = backpropagation_train(
    X_gate,
    Y_AND_2OUT,
    n_hidden=2,
    n_outputs=2,
    learning_rate=0.05,
    max_epochs=1000,
    convergence_error=0.002,
    seed=42,
    init_scale=0.5
)

print("Converged:", and_2out["converged"])
print("Epochs:", and_2out["epoch"])
print("Final SSE:", and_2out["errors"][-1])

print("\nOutput nodes [O1, O2]:")
print(and_2out["outputs"])

decoded = np.argmax(and_2out["outputs"], axis=1)
decoded_logic = decoded.copy()

print("\nDecoded logic output:")
print(decoded_logic)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(and_2out["errors"]) + 1), and_2out["errors"])
plt.axhline(0.002, linestyle="--", label="Threshold = 0.002")
plt.xlabel("Epoch")
plt.ylabel("SSE")
plt.title("A10 – Two-Output AND Network")
plt.grid(True)
plt.legend()
plt.show()


## A11 – AND and XOR using sklearn MLPClassifier

In [ ]:

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

and_mlp = MLPClassifier(
    hidden_layer_sizes=(2,),
    activation="logistic",
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)

and_mlp.fit(X_gate, Y_AND.astype(int))
and_mlp_pred = and_mlp.predict(X_gate)

print("AND – MLPClassifier")
print("Predictions:", and_mlp_pred)
print("Accuracy:", accuracy_score(Y_AND, and_mlp_pred))


xor_mlp = MLPClassifier(
    hidden_layer_sizes=(2,),
    activation="logistic",
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)

xor_mlp.fit(X_gate, Y_XOR.astype(int))
xor_mlp_pred = xor_mlp.predict(X_gate)

print("\nXOR – MLPClassifier")
print("Predictions:", xor_mlp_pred)
print("Accuracy:", accuracy_score(Y_XOR, xor_mlp_pred))


## A12 – Use MLPClassifier on your project dataset

In [ ]:



from google.colab import files

uploaded = files.upload()

project_file = next(iter(uploaded))
project_df = pd.read_csv(project_file)

print("Dataset shape:", project_df.shape)
print("\nColumns:")
print(project_df.columns.tolist())

display(project_df.head())


In [ ]:

TARGET_COLUMN = "target"

if TARGET_COLUMN not in project_df.columns:
    print("TARGET_COLUMN was not found.")
    print("Available columns are:")
    print(project_df.columns.tolist())
else:
    X_project = project_df.drop(columns=[TARGET_COLUMN]).copy()
    y_project = project_df[TARGET_COLUMN].copy()

   
    X_project = pd.get_dummies(X_project, drop_first=False)


    X_project = X_project.astype(float)

    if not pd.api.types.is_numeric_dtype(y_project):
        y_project = pd.factorize(y_project)[0]

    y_project = np.asarray(y_project)

    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    X_train, X_test, y_train, y_test = train_test_split(
        X_project.values,
        y_project,
        test_size=0.2,
        random_state=42,
        stratify=y_project if len(np.unique(y_project)) > 1 else None
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    project_mlp = MLPClassifier(
        hidden_layer_sizes=(16, 8),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=2000,
        random_state=42
    )

    project_mlp.fit(X_train, y_train)

    project_pred = project_mlp.predict(X_test)

    print("Project MLPClassifier Results")
    print("Test accuracy:", accuracy_score(y_test, project_pred))
    print("Iterations:", project_mlp.n_iter_)

    project_result = pd.DataFrame({
        "Actual": y_test,
        "Predicted": project_pred
    })

    display(project_result.head(20))
